In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 9.4 Measurement, Channels, and the Choi Matrix

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Chapter IX — Quantum Information as Linear Algebra",
    number="9.4",
    title="Measurement, Channels, and the Choi Matrix",
    blurb="Noise turns state vectors into density matrices and gates into "
    "channels — linear maps on matrices, which Chapter VI taught the course "
    "to write as matrices themselves. One reshape of that superoperator, "
    "the Choi matrix, answers the subject's deepest structural question "
    "with a psd test — and its most famous failure detects entanglement.",
    difficulty="advanced",
    estimate="120–150 min",
)

## Notebook overview

[§9.2](tensor-products-entanglement.ipynb)'s partial traces produced
matrices that were not rank-one projectors: **density matrices**, the
states of realistic, noisy systems. This notebook builds their calculus.
States become Hermitian psd trace-one matrices filling
[§9.1](qubits-gates-bloch.ipynb)'s Bloch *ball*; evolutions become
**channels** $\rho \mapsto \sum_k K_k\rho K_k^{\dagger}$; and because a
channel is a linear map on matrices, the course's
[§6.4](../06-structure/kronecker-vec-separable.ipynb) machinery writes it
as an ordinary matrix acting on $\operatorname{vec}(\rho)$ — gated here
to agree with the Kraus route *bitwise*, because the two spellings
perform identical arithmetic.

The centrepiece is the **Choi matrix**: feed half of a Bell pair through
the channel and the output's matrix decides, by a psd test, whether the
map is *completely* positive — physical on entangled inputs, not merely
on lone ones. The transpose map fails the test with an exact eigenvalue
of $-\tfrac12$, and that failure is a tool: the same negative eigenvalue,
read off a partially transposed state, is the **PPT criterion** that
certifies the Bell pair's entanglement. A no-go theorem and a detector,
one psd computation apart.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Nielsen and Chuang {cite}`nielsen2010` Chapters 2 and 8;
> the Choi correspondence is Choi's {cite}`choi1975`, the PPT criterion
> Peres's {cite}`peres1996`. The psd machinery is
> [§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb)'s, and the
> vec/Kronecker identities are
> [§6.4](../06-structure/kronecker-vec-separable.ipynb)'s.

## Theory in brief

### Density matrices: the Bloch ball filled in

A noisy preparation is an ensemble $\{p_k, |\psi_k\rangle\}$, and every
statistic it produces depends only on

```{math}
:label: eq-mc-density
\rho \;=\; \sum_k p_k\,|\psi_k\rangle\langle\psi_k| ,
\qquad \rho = \rho^{\dagger},\quad \rho \succeq 0,\quad
\operatorname{tr}\rho = 1 ,
```

with expectations $\langle A\rangle = \operatorname{tr}(\rho A)$. For
one qubit, $\rho = \tfrac12(I + \mathbf{r}\cdot\boldsymbol{\sigma})$
with $\lVert\mathbf{r}\rVert \le 1$: pure states on
[§9.1](qubits-gates-bloch.ipynb)'s sphere, mixtures strictly inside,
and purity $\operatorname{tr}\rho^2 = \tfrac12(1 +
\lVert\mathbf{r}\rVert^2)$ measuring the distance in.

### Channels: Kraus operators

The general physical evolution is a **channel**,

```{math}
:label: eq-mc-kraus
\Phi(\rho) \;=\; \sum_k K_k\,\rho\,K_k^{\dagger},
\qquad \sum_k K_k^{\dagger}K_k = I ,
```

the completeness condition making it trace-preserving. Unitaries are
the one-term case; measurement-and-forget maps, decay and dephasing all
take more terms, and each named channel acts on the Bloch ball by a
specific contraction the exercises gate against closed forms.

### The channel as one matrix

$\Phi$ is linear in $\rho$, so it *is* a matrix once $\rho$ is a
vector. With row-major `reshape(-1)` as vec,
[§6.4](../06-structure/kronecker-vec-separable.ipynb)'s identity
$\operatorname{vec}(AXB) = (A \otimes B^{\top})\operatorname{vec}(X)$
gives the **superoperator**

```{math}
:label: eq-mc-superop
S_{\Phi} \;=\; \sum_k K_k \otimes \overline{K_k},
\qquad
\operatorname{vec}(\Phi(\rho)) = S_{\Phi}\operatorname{vec}(\rho) ,
```

and channel composition becomes matrix multiplication — the course's
oldest theme, absorbing quantum dynamics.

### Choi: complete positivity is a psd test

Positivity of $\Phi$ on single systems is not enough: physics demands
$\Phi \otimes \mathrm{id}$ stay positive when the system is entangled
with a bystander. Feeding half a Bell pair through decides it all at
once. The **Choi matrix**

```{math}
:label: eq-mc-choi
J(\Phi) \;=\; (\Phi \otimes \mathrm{id})\,
|\Omega\rangle\langle\Omega| ,
\qquad |\Omega\rangle = \tfrac{1}{\sqrt2}(|00\rangle + |11\rangle) ,
```

is psd **exactly when** $\Phi$ is completely positive {cite}`choi1975`.
The transpose map is the canonical failure: positive on every single
system, yet $J$ picks up an eigenvalue $-\tfrac12$ — and reading the
same computation backwards, a state whose *partial transpose* has a
negative eigenvalue must be entangled: the PPT criterion
{cite}`peres1996`, with the Bell pair flagged at exactly $-\tfrac12$.

---
## Setup

Data and restated instruments only: the Pauli matrices, the maximally
entangled reference state, the three named channels' Kraus sets at
stated noise levels, and the seeded state factory. The Kraus engine,
the superoperator, the Choi construction and the PPT witness are all
built in the exercises, where they are the lesson.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ecp import validate
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random state below comes from this seed

EPS = np.finfo(float).eps

# data: the one-qubit cast, restated once more.
SIGMA_X = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
SIGMA_Y = np.array([[0.0, -1.0j], [1.0j, 0.0]])
SIGMA_Z = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)
PAULIS = [SIGMA_X, SIGMA_Y, SIGMA_Z]
I2 = np.eye(2, dtype=complex)

# data: the maximally entangled reference state of Eq. 4.
OMEGA = np.zeros(4, dtype=complex)
OMEGA[0] = OMEGA[3] = 1.0 / np.sqrt(2.0)

# data: the three named channels at stated noise levels — Kraus sets as
# the explicit constants every text tabulates.
P_DEPOL = 0.3
KRAUS_DEPOL = [np.sqrt(1 - 3 * P_DEPOL / 4) * I2,
               np.sqrt(P_DEPOL / 4) * SIGMA_X,
               np.sqrt(P_DEPOL / 4) * SIGMA_Y,
               np.sqrt(P_DEPOL / 4) * SIGMA_Z]
P_DEPH = 0.3
KRAUS_DEPH = [np.sqrt(1 - P_DEPH / 2) * I2,
              np.sqrt(P_DEPH / 2) * SIGMA_Z]
GAMMA_AMP = 0.4
KRAUS_AMP = [np.array([[1.0, 0.0], [0.0, np.sqrt(1 - GAMMA_AMP)]],
                      dtype=complex),
             np.array([[0.0, np.sqrt(GAMMA_AMP)], [0.0, 0.0]],
                      dtype=complex)]
CHANNELS = {"depolarizing": KRAUS_DEPOL, "dephasing": KRAUS_DEPH,
            "amplitude damping": KRAUS_AMP}


# instrument: the seeded state factory — Gaussian draws and a norm.
def random_state(generator, dim=2):
    """A random pure state on `dim` amplitudes: complex Gaussian, normalised."""
    v = generator.standard_normal(dim) + 1.0j * generator.standard_normal(dim)
    return v / np.linalg.norm(v)


# instrument: Bloch coordinates of a DENSITY matrix — the trace pairing,
# extraction not construction (9.1's reader, one rung up).
def bloch_of_rho(rho):
    """The Bloch vector tr(rho sigma_i) of a one-qubit density matrix."""
    return np.array([float(np.real(np.trace(rho @ s))) for s in PAULIS])

## Exercise 1: Density matrices fill the ball

{eq}`eq-mc-density` promotes states from vectors to matrices. The
tests are [§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb)'s,
and the geometry is [§9.1](qubits-gates-bloch.ipynb)'s sphere growing
an interior.

**Part a)** Write `mix(states, probs)` returning
$\sum_k p_k|\psi_k\rangle\langle\psi_k|$ via `np.outer`. For 100
seeded three-state mixtures (Dirichlet-free: seeded uniform weights,
normalised), gate the three defining properties: Hermitian to
$10^{-15}$, trace one to $10^{-14}$, smallest eigenvalue above
$-10^{-14}$ (one-sided psd).

**Part b)** Gate the geometry: every mixture's Bloch vector has
$\lVert\mathbf{r}\rVert \le 1 + 10^{-12}$, purity equals
$\tfrac12(1 + \lVert\mathbf{r}\rVert^2)$ to $10^{-13}$, and — the
strict part — each seeded *three-state* mixture of non-parallel states
lands **strictly inside** ($\lVert\mathbf{r}\rVert < 1 - 10^{-6}$,
one-sided over this seed's draws, reported): mixing is a convex
average in the ball, and averages of distinct points leave the sphere.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    worst_h < 1e-15 and worst_t < 1e-14 and worst_eig < 1e-14,
    "seeded mixtures are Hermitian, unit-trace and psd (Eq. 1)",
    f"100 three-state ensembles: 3.3's tests, passed at rounding — the "
    "state space of noisy preparations",
)
validate.check(
    worst_len < 1e-12 and worst_pur < 1e-13
    and max_inside < 1.0 - 1e-6,
    "and they fill the interior of the Bloch ball",
    f"||r|| <= 1 with purity = (1 + ||r||^2)/2 at {worst_pur:.1e}; the "
    f"longest arrow of this seed's mixtures is {max_inside:.4f} — convex "
    "averages of distinct pure points leave the sphere, one-sidedly "
    "reported for this seed",
)

## Exercise 2: Channels: the Kraus engine

{eq}`eq-mc-kraus` is the general physical map. This exercise builds
the engine, gates the algebra, and reads each named channel's
signature off the Bloch ball against its closed form.

**Part a)** Write `kraus_apply(Ks, rho)` returning
$\sum_k K_k\rho K_k^{\dagger}$. Gate the completeness sums
$\sum_k K_k^{\dagger}K_k = I$ for all three Setup channels to
$10^{-15}$, and trace preservation on 100 seeded mixtures to
$10^{-14}$.

**Write this one yourself** — one comprehension, and every noisy
evolution in the subject runs through it.

**Part b)** Gate the Bloch signatures to $10^{-12}$ on 50 seeded pure
states: depolarizing at $p = 0.3$ scales
$\mathbf{r} \mapsto (1-p)\mathbf{r}$ (isotropic shrink); dephasing at
$p = 0.3$ scales $(r_x, r_y)$ by $1-p$ and **keeps** $r_z$ (the
equator contracts, the axis survives); amplitude damping at
$\gamma = 0.4$ scales $(r_x, r_y)$ by $\sqrt{1-\gamma}$ and maps
$r_z \mapsto (1-\gamma)r_z + \gamma$ — an *affine* action, dragging
everything toward the ground-state pole. Draw the three images of the
sphere's $xz$ great circle.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    worst_comp < 1e-15 and worst_tp < 1e-14,
    "the Kraus sets are complete and the channels preserve trace (Eq. 2)",
    f"completeness at {worst_comp:.1e}, trace held to {worst_tp:.1e} over "
    "300 channel applications — probability goes nowhere",
)
validate.below(
    worst_sig, 1e-12,
    "each channel's Bloch action matches its closed form",
    "isotropic shrink, equator-only contraction, and the affine drag "
    "toward the pole — three kinds of noise, three signatures, 50 seeded "
    "states each",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3: The channel is literally a matrix

{eq}`eq-mc-superop` says a channel has a matrix the moment $\rho$
becomes a vector — [§6.4](../06-structure/kronecker-vec-separable.ipynb)'s
identity with a conjugate. The two routes perform the *same*
multiplications in the same order, so they agree bitwise, and the
course gates exactly that.

**Part a)** Write `superop(Ks)` returning
$\sum_k K_k \otimes \overline{K_k}$, acting on the row-major
`rho.reshape(-1)`. Gate the two routes on 100 seeded mixtures and all
three channels: `superop(Ks) @ vec(rho)` **equals**
`vec(kraus_apply(Ks, rho))` with `np.array_equal` — bitwise, because
$\operatorname{vec}(K\rho K^{\dagger}) = (K \otimes \overline{K})
\operatorname{vec}(\rho)$ is the same arithmetic re-parenthesised
only in the bookkeeping, not the floating-point operations. (If a
future BLAS reorders one side, the fallback claim is $10^{-15}$;
today the agreement is exact, and the gate says so.)

**Write this one yourself** — 6.4's vec identity, conjugated into
quantum service.

**Part b)** Gate composition-as-multiplication: dephasing *after*
amplitude damping, computed as nested `kraus_apply`, equals the single
matrix `superop(KRAUS_DEPH) @ superop(KRAUS_AMP)` acting on vec, to
$5\times10^{-15}$ on the same 100 mixtures — chained noise is a matrix
product, which is how long circuits are actually analysed.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    bitwise_ok or worst_fallback < 1e-15,
    "the superoperator route EQUALS the Kraus route (Eq. 3)",
    "bitwise today: kron-then-matvec performs the same multiplications "
    "as sandwich-then-reshape, so 6.4's identity holds without rounding "
    "— with a 1e-15 fallback stated in case a different BLAS reorders",
)
validate.below(
    worst_compose, 5e-15,
    "and chained noise is one matrix product",
    "dephasing-after-damping as nested sandwiches vs one 4x4 times vec: "
    "channel composition is matrix multiplication, the course's oldest "
    "sentence absorbing its newest subject",
)

## Exercise 4: Choi: complete positivity is one psd test

{eq}`eq-mc-choi` folds a channel's entire behaviour — on entangled
inputs included — into one $4\times4$ matrix. Positive semidefinite
means physical; and the transpose map shows the test has teeth.

**Part a)** Write `choi(Ks)` as $(\Phi \otimes \mathrm{id})$ applied
to $|\Omega\rangle\langle\Omega|$, i.e. `kraus_apply` with the lifted
operators $K_k \otimes I$. Gate for all three channels: $J$ Hermitian
to $10^{-15}$, trace one to $10^{-14}$, smallest eigenvalue above
$-10^{-14}$ — three psd verdicts, three physical channels.

**Part b)** The canonical failure: build the transpose map's Choi
matrix directly ($(\mathrm{T} \otimes \mathrm{id})
|\Omega\rangle\langle\Omega|$, a partial transpose of the reference
projector) and gate its spectrum equal to
$(-\tfrac12, \tfrac12, \tfrac12, \tfrac12)$ to $10^{-14}$ — an exact
fraction announcing that transposition is not completely positive.

**Part c)** Gate what makes the failure interesting: the transpose of
every one of 100 seeded single-qubit mixtures is still psd (smallest
eigenvalue above $-10^{-14}$) — the map is *positive*, and only the
Bell-pair test of Part b exposes it. Draw the four Choi spectra.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    worst_choi < 1e-14,
    "the three physical channels' Choi matrices are psd states (Eq. 4)",
    f"Hermitian, unit trace, smallest eigenvalue floored at "
    f"{worst_choi:.1e} — complete positivity, certified by 3.3's test on "
    "one 4x4 matrix per channel",
)
validate.below(
    trans_gap, 1e-14,
    "the transpose map's Choi spectrum is (-1/2, 1/2, 1/2, 1/2) exactly",
    "an exact fraction: transposition fails complete positivity by the "
    "largest margin a trace-one matrix allows",
)
validate.below(
    worst_pos, 1e-14,
    "yet the transpose is positive on every lone qubit",
    "100 seeded mixtures stay psd under transposition — the failure is "
    "invisible until entanglement enters, which is exactly what the "
    "Choi construction feeds it",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5: The failure, weaponised: PPT detects entanglement

Exercise 4's negative eigenvalue becomes a detector when read against
states instead of maps: if $\rho^{T_B}$ (transpose on one subsystem)
has a negative eigenvalue, $\rho$ is entangled — because separable
states are mixtures of products, and products transpose to products
{cite}`peres1996`.

**Part a)** Write `partial_transpose(rho4)` transposing the second
qubit's indices (a `reshape(2,2,2,2)`, one `transpose`, reshape back).
Gate the witness on the Bell state: the spectrum of
$|\Omega\rangle\langle\Omega|^{T_B}$ equals
$(-\tfrac12, \tfrac12, \tfrac12, \tfrac12)$ to $10^{-14}$ — maximal
entanglement, flagged at the maximal margin.

**Part b)** Gate that the witness spares what it must: for 100 seeded
*separable* mixtures $\sum_j p_j\,\rho_j^A \otimes \rho_j^B$ (three
products each), the partial transpose's smallest eigenvalue stays
above $-10^{-13}$ — no false alarms, one-sidedly, across this seed's
draws.

**Part c)** Gate the dial's threshold: for the Werner family
$\rho_w = w\,|\Omega\rangle\langle\Omega| + (1-w)\,I/4$, the
partial transpose's smallest eigenvalue is $(1 - 3w)/4$ — gate the
closed form to $10^{-13}$ on a 100-point sweep, so the witness fires
exactly for $w > 1/3$: entanglement survives depolarizing mixing down
to a sharp, exactly known threshold.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.below(
    bell_pt_gap, 1e-14,
    "the Bell pair's partial transpose carries the exact -1/2",
    "Exercise 4's no-go eigenvalue, read backwards as a detector: "
    "maximal entanglement flagged at the maximal margin",
)
validate.below(
    worst_sep, 1e-13,
    "and separable states never trip the witness (one-sided)",
    "100 seeded product mixtures stay psd under partial transposition — "
    "products transpose to products, so the alarm is honest",
)
validate.below(
    worst_werner, 1e-13,
    "the Werner dial's smallest PT eigenvalue is (1 - 3w)/4, exactly",
    "a closed-form threshold at w = 1/3: how much depolarizing mixing "
    "the Bell pair's detectable entanglement survives, known in "
    "fractions",
)

---
## Notebook summary

**The ball filled in.** 100 seeded three-state mixtures passed
[§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb)'s tests at
rounding, obeyed purity $= \tfrac12(1 + \lVert\mathbf{r}\rVert^2)$ to
$10^{-14}$-scale, and sat strictly inside the sphere (longest arrow
$0.9940$ for this seed) — density matrices as the convex hull
[§9.1](qubits-gates-bloch.ipynb)'s sphere never had.

**Three noises, three signatures.** The Kraus engine held completeness
and trace preservation at rounding; depolarizing shrank isotropically
by $0.7$, dephasing contracted only the equator, amplitude damping
contracted *and translated* toward the pole — every Bloch action within
$10^{-13}$-scale of its closed form.

**A channel is literally a matrix.** The superoperator
$\sum K \otimes \overline{K}$ agreed with the Kraus route **bitwise**
on all 300 applications — same multiplications, different bookkeeping —
and chained noise collapsed to one matrix product at $10^{-16}$-scale:
[§6.4](../06-structure/kronecker-vec-separable.ipynb)'s vec identity in
quantum service.

**One psd test decides physicality, and its failure detects
entanglement.** The three channels' Choi matrices were psd states; the
transpose map's spectrum came out $(-\tfrac12, \tfrac12, \tfrac12,
\tfrac12)$ exactly while remaining positive on every lone qubit; and
the same $-\tfrac12$, read off the Bell pair's partial transpose,
certified its entanglement — with 100 separable mixtures raising no
false alarm and the Werner dial's threshold landing on $w = 1/3$ by the
closed form $(1 - 3w)/4$.

**Methods introduced.** `mix`, `kraus_apply`, the closed-form Bloch
signatures, `superop` (6.4's vec identity conjugated), `choi` via
lifted Kraus operators, `partial_transpose`, and the Werner-family
threshold as an exact-fraction gate.

## Outlook

- **The QFT closes the chapter.**
  [§9.5](quantum-fourier-transform.ipynb) returns to unitary — noiseless
  — evolution, where the chapter's last handshake waits:
  [§6.3](../06-structure/circulant-toeplitz-fft.ipynb)'s DFT matrix as
  the quantum computer's workhorse transform.
- **PPT is necessary, not sufficient, beyond 2x3.** For two qubits the
  witness is exact; at higher dimensions entangled states exist with
  positive partial transpose ("bound" entanglement), and detecting them
  needs the full apparatus of entanglement witnesses — Hermitian
  operators whose expectation separates the separable convex set from a
  target state, i.e. [§2.1](../02-orthogonality/projections-normal-equations.ipynb)-style
  separating hyperplanes in matrix space.
- **Master equations.** Continuous-time noise exponentiates a Lindblad
  generator: the channel semigroup $e^{t\mathcal{L}}$, with
  [§3.6](../03-eigenvalues/matrix-functions-exponential.ipynb)'s matrix
  exponential acting on the superoperator of Exercise 3.
- **Error correction.** The Kraus picture is where stabilizer codes
  live: a code is a subspace on which the noise's Kraus operators act
  invertibly, and the Knill–Laflamme conditions are one more exercise
  in projectors and inner products.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()